<a href="https://colab.research.google.com/github/fvarellalopes/ideogramoss-colab/blob/main/Ideogram4_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq && apt-get install -y -qq wget
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared

In [ ]:
!pip install -q uvicorn fastapi pydantic pillow fire --no-cache-dir

In [ ]:
import os
from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN') or os.environ.get('HF_TOKEN', '')
IDEO_KEY = userdata.get('IDEOGRAM_API_KEY') or os.environ.get('IDEOGRAM_API_KEY', '')


In [ ]:
!rm -rf /content/ideogram4
!git clone --depth=1 https://github.com/fvarellalopes/ideogramoss-colab.git /content/ideogram4
!cd /content/ideogram4 && pip install -q -e .


In [ ]:
import os
import sys
import torch

if HF_TOKEN:
    os.environ.setdefault('HF_TOKEN', HF_TOKEN)

repo_src = '/content/ideogram4/src'

# Remove caminhos antigos/duplicados do ideogram4 para evitar cache errado
sys.path = [p for p in sys.path if 'ideogram4' not in p]
sys.path.insert(0, repo_src)

print('Usando ideogram4 em:', repo_src)

try:
    from ideogram4 import Ideogram4Pipeline, Ideogram4PipelineConfig
    print('Import ok direto de ideogram4')
except Exception as e_direct:
    print('Import direto falhou:', e_direct)
    try:
        from ideogram4.pipeline_ideogram4 import Ideogram4Pipeline, Ideogram4PipelineConfig
        print('Import ok via pipeline_ideogram4')
    except Exception as e_fallback:
        raise ImportError(f'Falhou ambos imports: {e_direct}; {e_fallback}')

SEED = 42

print('Loading Ideogram 4 (nf4) on cuda...')
pipe = Ideogram4Pipeline.from_pretrained(
    config=Ideogram4PipelineConfig(
        weights_repo='ideogram-ai/ideogram-4-nf4',
        colab_friendly=False,
    ),
    device='cuda',
    dtype=torch.bfloat16,
)
pipe.eval()
print('Pipeline pronta')


In [ ]:
from IPython.display import display
prompt = 'A cinematic portrait of a mage with cybernetic implants, neon circuit tattoos, bionic arm servos glowing blue, standing in a cyberpunk city at night. High detail, photorealistic.'
images = pipe(
    prompt,
    height=640,
    width=512,
    num_steps=12,
    guidance_scale=0.0,
    seed=SEED,
)
img = images[0]
display(img)

In [ ]:
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import uvicorn
import threading
import time
import torch
import base64
from io import BytesIO

app = FastAPI()
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_credentials=True,
    allow_methods=['*'],
    allow_headers=['*'],
)

TOKEN = 'YOUR_SECRET_TOKEN'

class Req(BaseModel):
    prompt: str
    token: str
    height: int = 1024
    width: int = 1024
    seed: int | None = None

@app.get('/health')
async def health():
    return {'status': 'ok'}

@app.post('/generate')
async def generate(req: Req):
    if req.token != TOKEN:
        raise HTTPException(status_code=401, detail='Invalid Token')
    seed = req.seed if req.seed is not None else torch.randint(0, 2**32 - 1, (1,)).item()
    images = pipe(
        req.prompt,
        height=req.height,
        width=req.width,
        num_steps=12,
        guidance_scale=7.0,
        seed=seed,
    )
    buf = BytesIO()
    images[0].save(buf, format='PNG')
    return {'image': base64.b64encode(buf.getvalue()).decode()}

def run_server():
    uvicorn.run(app, host='0.0.0.0', port=8081, log_level='info')

try:
    import subprocess as _subprocess
    _subprocess.run(['bash', '-lc', 'fuser -k 8081/tcp || true'], check=False)
except Exception:
    pass

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(2)
print('Servidor Ideogram4 na porta 8081')


In [ ]:
import subprocess, re, time

cloudflared_proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:8081'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
)

tunnel_url = None
print('Esperando URL do tunnel...')
for i in range(60):
    raw = cloudflared_proc.stdout.readline()
    line = raw.decode('utf-8', errors='replace') if raw else ''
    if not line:
        time.sleep(0.5)
        continue
    print(line.strip())
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        tunnel_url = m.group(0)
        break

if tunnel_url:
    print(f'TUNEL: {tunnel_url}')
    print(f'Endpoint: {tunnel_url}/generate')
else:
    print('Nao encontrou URL do tunnel')
